In [ ]:
import pandas as pd
data = pd.read_excel("British Airways.xlsx")

df = pd.DataFrame(data)
df["FLIGHT_DATE"] = pd.to_datetime(df["FLIGHT_DATE"])
df["TOTAL_SEATS"] = df[["FIRST_CLASS_SEATS", "BUSINESS_CLASS_SEATS", "ECONOMY_SEATS"]].sum(axis=1)

# ---------- STEP 2: DEFINE CATEGORIES ----------
# Category = Haul (SHORT/LONG) x Time of day
df["CATEGORY"] = df["HAUL"] + " | " + df["TIME_OF_DAY"]

# ---------- STEP 3a: ESTIMATE ELIGIBILITY % PER CATEGORY ----------
tiers = {"TIER1": "TIER1_ELIGIBLE_PAX", "TIER2": "TIER2_ELIGIBLE_PAX", "TIER3": "TIER3_ELIGIBLE_PAX"}
g = df.groupby("CATEGORY")
lookup = pd.DataFrame({"FLIGHTS": g.size(), "AVG_SEATS": g["TOTAL_SEATS"].mean().round(0)})
for t, col in tiers.items():
    lookup[f"{t}_PCT"] = (g[col].sum() / g["TOTAL_SEATS"].sum()).round(4)
lookup = lookup.reset_index()
print("=== REUSABLE LOOKUP TABLE (% of seats eligible per tier) ===")
print(lookup.to_string(index=False))

# ---------- STEP 3b: APPLY TO A REPRESENTATIVE SAMPLE ----------

sample = df[(df["FLIGHT_DATE"] >= "2025-07-07") & (df["FLIGHT_DATE"] <= "2025-07-13")].copy()
print(f"\nSample: {len(sample)} flights, {sample.FLIGHT_DATE.min().date()} to {sample.FLIGHT_DATE.max().date()}")

def apply_lookup(schedule, lookup):
    """Reusable: works on any future schedule with HAUL, TIME_OF_DAY and seat columns."""
    schedule = schedule.copy()
    schedule["TOTAL_SEATS"] = schedule[["FIRST_CLASS_SEATS", "BUSINESS_CLASS_SEATS", "ECONOMY_SEATS"]].sum(axis=1)
    schedule["CATEGORY"] = schedule["HAUL"] + " | " + schedule["TIME_OF_DAY"]
    out = schedule.merge(lookup.drop(columns=["FLIGHTS", "AVG_SEATS"]), on="CATEGORY", how="left")
    if out["TIER1_PCT"].isna().any():
        print("WARNING: categories missing from lookup:", out.loc[out.TIER1_PCT.isna(), "CATEGORY"].unique())
    for t in tiers:
        out[f"{t}_EST_PAX"] = (out["TOTAL_SEATS"] * out[f"{t}_PCT"]).round(0)
    return out

est = apply_lookup(sample, lookup)

summary = est.groupby("CATEGORY").agg(
    FLIGHTS=("FLIGHT_NO", "count"), TOTAL_SEATS=("TOTAL_SEATS", "sum"),
    TIER1_EST_PAX=("TIER1_EST_PAX", "sum"), TIER2_EST_PAX=("TIER2_EST_PAX", "sum"),
    TIER3_EST_PAX=("TIER3_EST_PAX", "sum")).reset_index()
summary.loc[len(summary)] = ["TOTAL"] + summary.iloc[:, 1:].sum().tolist()
print("\n=== ESTIMATED LOUNGE USERS BY CATEGORY (sample week) ===")
print(summary.to_string(index=False))

# ---------- SANITY CHECK vs the dataset's own tier columns ----------
actual = sample[list(tiers.values())].sum()
print("\nEstimated vs dataset values (sample):")
for t, col in tiers.items():
    print(f"{t}: estimated {est[f'{t}_EST_PAX'].sum():.0f} | dataset {actual[col]}")

# ---------- EXPORT ----------
lookup.to_csv("lounge_lookup_table.csv", index=False)
summary.to_csv("lounge_sample_estimates.csv", index=False)
try:
    files.download("lounge_lookup_table.csv"); files.download("lounge_sample_estimates.csv")
except NameError:
    pass

=== REUSABLE LOOKUP TABLE (% of seats eligible per tier) ===
         CATEGORY  FLIGHTS  AVG_SEATS  TIER1_PCT  TIER2_PCT  TIER3_PCT
 LONG | Afternoon      939      293.0     0.0020     0.0277     0.1052
   LONG | Evening     1190      292.0     0.0021     0.0267     0.1026
 LONG | Lunchtime      435      290.0     0.0018     0.0271     0.1038
   LONG | Morning     1461      292.0     0.0021     0.0279     0.1063
SHORT | Afternoon     1366      180.0     0.0034     0.0431     0.1659
  SHORT | Evening     1783      180.0     0.0033     0.0445     0.1700
SHORT | Lunchtime      757      180.0     0.0039     0.0455     0.1728
  SHORT | Morning     2069      180.0     0.0034     0.0436     0.1674

Sample: 314 flights, 2025-07-07 to 2025-07-13

=== ESTIMATED LOUNGE USERS BY CATEGORY (sample week) ===
         CATEGORY  FLIGHTS  TOTAL_SEATS  TIER1_EST_PAX  TIER2_EST_PAX  TIER3_EST_PAX
 LONG | Afternoon     32.0       8781.0           18.0          245.0          928.0
   LONG | Evening     39.